# SHAP + LIME explainability comparison

This notebook reconstructs the historical Wine/MLP explainability exercise with deterministic class alignment and current APIs. SHAP and LIME explain model behavior; they do not prove that the model or its features are causally correct.


In [ ]:
from xai_portfolio.data import split_wine
from xai_portfolio.modeling import build_mlp, evaluate_classifier

split = split_wine()
model = build_mlp()
model.fit(split.X_train, split.y_train)
metrics = evaluate_classifier(model, split.X_test, split.y_test)
metrics["accuracy"], metrics["balanced_accuracy"], metrics["macro_f1"], metrics["labels"]


## Choose one explicit model output

Both explanation methods must refer to the same class. Here we use the model's `class_0` output and resolve its numeric index from `model.classes_` instead of hard-coding an assumed ordering.


In [ ]:
target_class = "class_0"
class_index = list(model.classes_).index(target_class)
row = split.X_test.iloc[[0]]
target_class, class_index, model.predict(row)[0], model.predict_proba(row)[0]


## SHAP

Use a bounded training background and a model-agnostic permutation explainer. This avoids passing the full training set to the historical `KernelExplainer` workflow.


In [ ]:
from xai_portfolio.attribution import bounded_background, shap_permutation_explanation, mean_abs_shap

background = bounded_background(split.X_train, max_rows=40)
shap_local = shap_permutation_explanation(model, background, row)
shap_local.values.shape


For a global class-specific ranking, explain a bounded test subset and take mean absolute attribution for the same output class.


In [ ]:
test_subset = split.X_test.iloc[:12]
shap_global = shap_permutation_explanation(model, background, test_subset)
shap_importance = mean_abs_shap(
    shap_global.values,
    split.X_train.columns,
    class_index=class_index,
)
shap_importance.head(8)


## LIME

LIME remains an optional dependency. The reconstruction reads `Explanation.as_map()` feature indices directly rather than parsing human-readable interval strings.


In [ ]:
from lime.lime_tabular import LimeTabularExplainer
from xai_portfolio.attribution import lime_weights_by_feature, mean_abs_lime

lime_explainer = LimeTabularExplainer(
    training_data=split.X_train.to_numpy(),
    feature_names=split.X_train.columns.tolist(),
    class_names=model.classes_.tolist(),
    mode="classification",
    discretize_continuous=True,
    random_state=42,
)

local_lime = lime_explainer.explain_instance(
    row.iloc[0].to_numpy(),
    model.predict_proba,
    labels=(class_index,),
    num_features=split.X_train.shape[1],
)
lime_weights_by_feature(
    local_lime,
    split.X_train.columns,
    class_index=class_index,
).sort_values(key=abs, ascending=False).head(8)


Aggregate several LIME explanations for the same class. `as_map()` preserves feature indices even when LIME displays discretized conditions such as ranges.


In [ ]:
lime_explanations = [
    lime_explainer.explain_instance(
        split.X_test.iloc[i].to_numpy(),
        model.predict_proba,
        labels=(class_index,),
        num_features=split.X_train.shape[1],
    )
    for i in range(min(12, len(split.X_test)))
]

lime_importance = mean_abs_lime(
    lime_explanations,
    split.X_train.columns,
    class_index=class_index,
)
lime_importance.head(8)


## Compare rankings

Feature-rank agreement is useful as an explanation-stability signal, but disagreement does not automatically mean one method is wrong: SHAP and LIME use different perturbation/background assumptions.


In [ ]:
from xai_portfolio.attribution import top_k_overlap

top_k_overlap(shap_importance, lime_importance, k=5)


## Interpretation boundary

Before trusting either explanation, validate the predictive model, data provenance, leakage risk, subgroup behavior, and domain assumptions. Explainability is a diagnostic lens on a model—not a substitute for model validation.
